# New-view → image-model → training-frameset, with exhaustive camera+depth checks

**Workflow this notebook models (no round-trip to Lyra):**

1. Build a **base scene** of splats from a clip (fresh DA3) — this is the existing
   training **frameset + cameras**, living in world frame **W**.
2. Author a **new camera B** in W and **render the splats** from it (RGB + depth + alpha).
3. Run the rendered RGB through an **image model** (inpaint/generate) to fill the
   **disoccluded** region the splats don't cover.
4. **Append** `(rgb, K, c2w, depth)` for B straight into the **frameset + cameras**
   (`VideoData.append_frame`, what `SplatTrainer.append_supplied_frames` calls) so the
   photometric loop can train on it.

**The thing that must be right:** B's **camera** and **depth** must live in the **same
coordinate frame W** as every other frame. This notebook checks that *exhaustively*:

- **Camera** — `w2c == inv(c2w)` (OpenCV), viser↔OpenCV round-trips with no basis flip,
  intrinsics `K` are self-consistent (true K vs the `fov=60°` hardcode).
- **Seen-region depth** — the splat-rendered depth back-projects **exactly onto the
  existing splats** (median nearest-point ≈ 0): proof camera+depth are in W.
- **Painted-region depth (the crux)** — the splat render gives **depth=0** where α=0, so
  the inpainted pixels would **collapse onto the camera origin**. We recover their depth
  with **DA3**, **scale it on the overlap** with the existing splats, build a **composite
  depth**, and verify it is finite, in front of the camera, **seam-continuous**, and lands
  on/near the scene.
- **After append** — re-verify straight from the **stored frameset** using the trainer's
  *own* unprojection math, plus `train_mask` coverage and a seed-style unprojection.

> Run with the **`lyra2`** conda env (DA3 needs the libcudart shim). This notebook reuses
> repo code from `splat_trainer`, `viewer`, `depth_utils`, `lyra2_zoomgs_inference`.

## Notation — what **W** and **K** are

These two terms carry the whole notebook, so pin them down first.

- **W — the world frame.** The single 3-D coordinate system that *every* frame, camera,
  and splat lives in. DA3 picks W when it reconstructs the base clip: it chooses a reference
  view and expresses all camera poses **and** all depth points relative to it, in its own
  arbitrary (non-metric) scale. "The new camera/depth must match the coordinate frame of the
  other frames" means exactly: B's pose and its back-projected depth must be expressed in
  **this same W, at the same scale**. The base splats, the base cameras, the new camera B,
  and B's appended depth are all in W — that is the invariant every check defends.
  - **c2w / w2c** — a camera's pose as a 4×4 matrix. `c2w` (camera→world) maps a point from
    the camera's local axes into W; its translation column `c2w[:3,3]` is the camera **centre**
    in W. `w2c = inv(c2w)` is the inverse (world→camera) and is what gsplat/DA3 store. OpenCV
    convention: camera looks down **+Z**, x right, y down. Checking `w2c == inv(c2w)` just
    confirms the two stored forms are consistent.

- **K — the camera intrinsics.** The 3×3 pinhole matrix
  `[[fx, 0, cx], [0, fy, cy], [0, 0, 1]]` that maps a 3-D camera-space point to a pixel:
  `u = fx·X/Z + cx`, `v = fy·Y/Z + cy`. `fx, fy` are focal lengths **in pixels** (fixed by the
  field of view and image size), `cx, cy` the principal point (≈ image centre). K is what turns
  a pixel **+ depth** back into a 3-D ray when we *unproject* (`cam = K⁻¹·[u,v,1]·z`). If the K
  you **store** for a frame disagrees with the K that actually produced its pixels, every
  unprojected point lands on the wrong ray and the geometry shears out of W — which is why
  Section 3b pits the true K against the `fov=60°` hardcode (`demo.py:844`, `inpainter.py:247`).

In short: **W** answers *where/what scale*, **K** answers *how pixels map to rays*. Get both
right for the new frame and its geometry lines up with everything already in the scene.

In [ ]:
import os, sys, math
from types import SimpleNamespace
from pathlib import Path
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)

ROOT    = "/home/kristofe/Documents/Projects/lyra/Lyra-2"
DA3_SRC = os.path.join(ROOT, "lyra_2/_src/inference/depth_anything_3/src")
for p in (DA3_SRC, os.path.join(ROOT, "visergui"), ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)
os.makedirs(os.path.join(ROOT, "visergui", "notebooks", "_scratch"), exist_ok=True)
os.chdir(os.path.join(ROOT, "visergui", "notebooks", "_scratch"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)
np.random.seed(0); torch.manual_seed(0)

# gsplat may JIT-rebuild; its host compile needs cuda_runtime_api.h from
# $PREFIX/targets/.../include. Prepend to CPATH BEFORE importing gsplat.
_cuda_inc = os.path.join(sys.prefix, "targets", "x86_64-linux", "include")
if os.path.isdir(_cuda_inc):
    os.environ["CPATH"] = _cuda_inc + os.pathsep + os.environ.get("CPATH", "")

import gsplat
import scipy.spatial
import scipy.ndimage as ndi
import splat_trainer as st
from splat_trainer import VideoData, _per_frame_train_mask, _estimate_depth_scale
from lyra_2._src.inference.depth_utils import (
    load_da3_model, load_moge_model, moge_infer_depth_intrinsics)
from lyra_2._src.inference.lyra2_zoomgs_inference import _da3_infer_depth_intrinsics_single
print("device:", device, "| torch:", torch.__version__)

C0 = 0.28209479177387814  # SH DC constant (matches splat_trainer._build_initial_gaussians)

In [ ]:
# ----------------------------- helpers: render / unproject / metrics -----------
def pix_grid(H, W):
    ii, jj = torch.meshgrid(torch.arange(H, device=device),
                            torch.arange(W, device=device), indexing="ij")
    return torch.stack([jj, ii, torch.ones_like(ii)], -1).float()      # (H,W,3) [u,v,1]

def render_scene(scene, w2c, K, H, W):
    # gsplat RGB+ED -> rgb_uint8, depth=ED/alpha, alpha. Mirrors inpainter._render_splats.
    w2c = torch.as_tensor(w2c, dtype=torch.float32, device=device).reshape(1, 4, 4)
    K   = torch.as_tensor(K,   dtype=torch.float32, device=device).reshape(1, 3, 3)
    sh = scene.sh; sh_deg = int(round(sh.shape[1] ** 0.5)) - 1
    out, alpha, _ = gsplat.rasterization(
        means=scene.means, quats=torch.nn.functional.normalize(scene.quats, dim=-1),
        scales=scene.scales, opacities=scene.opacities, colors=sh,
        viewmats=w2c, Ks=K, width=int(W), height=int(H),
        sh_degree=sh_deg, packed=False, render_mode="RGB+ED")
    a = alpha[0, :, :, 0].clamp(0, 1)
    rgb = (out[0, :, :, :3].clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
    depth = (out[0, :, :, 3] / a.clamp_min(1e-6)).cpu().numpy()
    return rgb, depth, a.cpu().numpy()

def unproject(depth_hw, K_33, c2w_44):
    # VERBATIM math of splat_trainer._build_initial_gaussians (L462-470) and
    # _seed_splats_for_new_frames: cam = Kinv @ [u,v,1] * z ; world = R @ cam + t.
    depth_hw = torch.as_tensor(depth_hw, dtype=torch.float32, device=device)
    K   = torch.as_tensor(K_33,  dtype=torch.float32, device=device)
    c2w = torch.as_tensor(c2w_44, dtype=torch.float32, device=device)
    H, W = depth_hw.shape
    cam = torch.einsum("ij,hwj->hwi", torch.linalg.inv(K), pix_grid(H, W)) * depth_hw[..., None]
    return torch.einsum("ij,hwj->hwi", c2w[:3, :3], cam) + c2w[:3, 3]

def nn_dist(query_pts, ref_pts, k_ref=200_000, k_q=20_000):
    # nearest-neighbour distance from each query point to the reference cloud.
    ref = np.asarray(ref_pts).reshape(-1, 3); ref = ref[np.isfinite(ref).all(1)]
    q   = np.asarray(query_pts).reshape(-1, 3); q = q[np.isfinite(q).all(1)]
    if ref.shape[0] == 0 or q.shape[0] == 0:
        return np.array([np.nan])
    if ref.shape[0] > k_ref: ref = ref[np.random.choice(ref.shape[0], k_ref, replace=False)]
    if q.shape[0]   > k_q:   q   = q[np.random.choice(q.shape[0],   k_q,   replace=False)]
    d, _ = scipy.spatial.cKDTree(ref).query(q)
    return d

def fit_overlap_scale(splat_depth, da3_depth, overlap_mask):
    # median(splat_z / da3_z) on the overlap -> multiply DA3 depth by this to reach W units.
    r = splat_depth[overlap_mask] / np.clip(da3_depth[overlap_mask], 1e-6, None)
    return float(np.median(r))

def build_composite_depth(splat_depth, alpha, da3_depth, s, alpha_thresh=0.5):
    # seen pixels keep splat depth (already in W); painted/disoccluded pixels (alpha<thr)
    # get DA3 depth * overlap-scale. Returns (composite, seen_mask, hole_mask).
    seen = alpha >= alpha_thresh; hole = ~seen
    comp = splat_depth.copy(); comp[hole] = da3_depth[hole] * s
    return comp, seen, hole

def pose_errors(c2w_rec, c2w_true):
    a = np.asarray(c2w_rec, np.float64); b = np.asarray(c2w_true, np.float64)
    pos = float(np.linalg.norm(a[:3, 3] - b[:3, 3]))
    Rrel = a[:3, :3] @ b[:3, :3].T
    ang = math.degrees(math.acos(max(-1.0, min(1.0, (np.trace(Rrel) - 1.0) / 2.0))))
    return pos, ang

def show_overlay(groups, title, cams=None, n=4000, elev=18, azim=-72):
    fig = plt.figure(figsize=(7.5, 6.2)); ax = fig.add_subplot(111, projection="3d")
    for pts, color, label in groups:
        p = np.asarray(pts).reshape(-1, 3); p = p[np.isfinite(p).all(1)]
        if p.shape[0] > n: p = p[np.random.choice(p.shape[0], n, replace=False)]
        ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=1.0, c=color, label=label, alpha=0.5)
    for c2w, col, lab in (cams or []):
        cc = np.asarray(c2w)[:3, 3]
        ax.scatter([cc[0]], [cc[1]], [cc[2]], s=70, marker="^", c=col, label=lab, edgecolors="k")
    ax.set_title(title); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.legend(loc="upper right", fontsize=7); ax.view_init(elev=elev, azim=azim)
    try: ax.set_box_aspect((1, 1, 1))
    except Exception: pass
    plt.tight_layout(); plt.show()
print("helpers ready")

In [ ]:
# ----------------------------- helpers: frameset (VideoData) + scene -----------
def da3_pack(model, paths):
    # Same packing as splat_trainer._da3_inference_on_paths, but reuses a preloaded
    # model and returns a REAL VideoData (the trainer's frameset/cameras type).
    pred = model.inference(image=list(paths), process_res=504,
                           process_res_method="upper_bound_resize")
    imgs  = torch.from_numpy(pred.processed_images).to(device)        # (N,H,W,3) uint8
    depth = torch.from_numpy(pred.depth).to(device)                  # (N,H,W)
    K     = torch.from_numpy(pred.intrinsics).to(device)            # (N,3,3)
    w2c34 = torch.from_numpy(pred.extrinsics).to(device)            # (N,3,4) OpenCV w2c
    N, H, W, _ = imgs.shape
    w2c = torch.eye(4, device=device).expand(N, 4, 4).clone(); w2c[:, :3, :4] = w2c34
    conf = getattr(pred, "conf", None); conf = getattr(pred, "confidence", None) if conf is None else conf
    sky  = getattr(pred, "sky", None)
    return VideoData(
        rgb=imgs.float() / 255.0, depth=depth, K=K, w2c=w2c, c2w=torch.linalg.inv(w2c),
        conf=(torch.from_numpy(conf).to(device) if conf is not None else None),
        sky=(torch.from_numpy(sky).to(device).bool() if sky is not None else None),
        N=N, H=H, W=W)

def make_scene(data, max_points=500_000, conf_q=0.5, remove_sky=True):
    # Unproject the frameset's RGBD into a renderable gsplat scene (scales/opacities
    # ALREADY activated). Mirrors splat_trainer._build_initial_gaussians, minus .ply writes.
    H, W = int(data.H), int(data.W)
    Kinv = torch.linalg.inv(data.K)
    cam  = torch.einsum("nij,hwj->nhwi", Kinv, pix_grid(H, W)) * data.depth[..., None]
    world = torch.einsum("nij,nhwj->nhwi", data.c2w[:, :3, :3], cam) + data.c2w[:, :3, 3][:, None, None, :]
    valid = data.depth > 0
    if getattr(data, "sky", None) is not None and remove_sky:
        valid &= ~data.sky.bool()
    if getattr(data, "conf", None) is not None:
        cf = data.conf.flatten().float()
        step = max(1, cf.numel() // 2_000_000)
        thr = torch.quantile(cf[::step], conf_q)
        valid &= data.conf > thr
    pts  = world[valid]; cols = data.rgb[valid]; z = data.depth[valid]
    fx   = data.K[:, 0, 0][:, None, None].expand(data.N, H, W)[valid]
    n_valid = int(pts.shape[0])
    if n_valid > max_points:
        sel = torch.randperm(n_valid, device=device)[:max_points]
        pts, cols, z, fx = pts[sel], cols[sel], z[sel], fx[sel]
    ratio = max(1.0, n_valid / max(1, pts.shape[0]))
    tex = (z / fx * (ratio ** 0.5)).clamp_min(1e-4)
    M = int(pts.shape[0])
    scene = SimpleNamespace(
        means=pts.contiguous().float(),
        quats=torch.tensor([1., 0, 0, 0], device=device).expand(M, 4).contiguous(),
        scales=tex[:, None].expand(M, 3).contiguous().float(),
        opacities=torch.full((M,), 0.9, device=device),
        sh=((cols - 0.5) / C0)[:, None, :].contiguous().float())
    scene_scale = float(pts.std(dim=0).mean().item())
    return scene, scene_scale
print("frameset + scene helpers ready")

## Section 1 — Base frameset (the existing world W)

Extract frames from a clip, run DA3, pack into a **real `VideoData`** (the trainer's
frameset + cameras), and unproject into a renderable scene. This DA3-native frame **W**
is the ground truth every later camera/point is judged against.

In [ ]:
VIDEO    = os.path.join(ROOT, "assets/ours/museum_fwd.mp4")
N_FRAMES = 10
DA3_NAME = "depth-anything/DA3NESTED-GIANT-LARGE-1.1"

da3_model = load_da3_model(da3_model_name=DA3_NAME, da3_model_path_custom=None, device=str(device))

frames_dir = os.path.join(os.getcwd(), "frames_base")
base_paths = st._extract_frames(Path(VIDEO), Path(frames_dir), N_FRAMES)
base_data  = da3_pack(da3_model, base_paths)            # REAL VideoData = training frameset/cameras
base_scene, scene_scale = make_scene(base_data)
base_np    = base_scene.means.cpu().numpy()
print(f"frameset: {type(base_data).__name__}  N={base_data.N}  HxW={base_data.H}x{base_data.W}  "
      f"gaussians={base_np.shape[0]}  scene_scale={scene_scale:.3f}")
print("base camera centers (world W):\n", base_data.c2w[:, :3, 3].cpu().numpy().round(3))

## Section 2 — A new camera B in W, render the splats

The user authors **B** directly (known pose), here a translate+yaw offset of an existing
camera so it overlaps the scene yet exposes a **disocclusion** the splats can't cover.
**K_B** is B's true intrinsics. `render_scene` returns RGB + depth + alpha; α<0.5 marks
the hole the image model will fill (gsplat reports **depth=0** there).

In [ ]:
A_IDX         = base_data.N // 2
BASELINE_FRAC = 0.25                  # B offset as a fraction of scene_scale
ALPHA_THRESH  = 0.5                   # alpha below this = disoccluded (to be painted)
c2w_A = base_data.c2w[A_IDX].clone()
K_B   = base_data.K[A_IDX].clone()    # B's TRUE intrinsics (the camera we author)
H_B, W_B = base_data.H, base_data.W

def make_B(c2w_A, baseline_frac, yaw_deg=12.0):
    # Translate along camera right(+x) & forward(+z), and yaw about up(-y) to force a
    # disocclusion the splats can't cover. All in OpenCV camera axes.
    c2w_B = c2w_A.clone()
    right, up, fwd = c2w_A[:3, 0], c2w_A[:3, 1], c2w_A[:3, 2]
    d = baseline_frac * scene_scale
    c2w_B[:3, 3] = c2w_A[:3, 3] + d * right + 0.2 * d * fwd
    if abs(yaw_deg) > 1e-6:
        th = math.radians(yaw_deg); axis = up / up.norm()
        Kx = torch.tensor([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]],
                          device=device, dtype=torch.float32)
        Rd = torch.eye(3, device=device) + math.sin(th) * Kx + (1 - math.cos(th)) * (Kx @ Kx)
        c2w_B[:3, :3] = Rd @ c2w_A[:3, :3]
    return c2w_B

c2w_B = make_B(c2w_A, BASELINE_FRAC)
w2c_B = torch.linalg.inv(c2w_B)
rgb_B, depth_B, alpha_B = render_scene(base_scene, w2c_B, K_B, H_B, W_B)
seen = alpha_B >= ALPHA_THRESH
hole = ~seen
print(f"new cam B: baseline={BASELINE_FRAC*scene_scale:.3f}  seen={seen.mean():.1%}  hole={hole.mean():.1%}")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].imshow(rgb_B); ax[0].set_title("splat render @ B (RGB)")
ax[1].imshow(hole, cmap="gray"); ax[1].set_title(f"disocclusion (alpha<{ALPHA_THRESH}) = {hole.mean():.0%}")
im = ax[2].imshow(depth_B, cmap="turbo"); ax[2].set_title("splat depth (0 in hole)"); fig.colorbar(im, ax=ax[2], shrink=0.8)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Section 3 — Check B's CAMERA is in world W

Three independent proofs: **(a)** `w2c == inv(c2w)` in the stored OpenCV convention;
**(b)** the viser↔OpenCV helper the demo uses round-trips with no basis flip; **(c)** B's
**seen-region** rendered depth back-projects **onto the existing splats** — the strongest
proof that B's camera + seen depth share frame W. Section 3b then shows how a wrong
intrinsics (the `fov=60°` hardcode) shears the same geometry out of W.

In [ ]:
# (a) OpenCV inverse round-trip in the convention we store.
err_inv = float((w2c_B - torch.linalg.inv(c2w_B)).abs().max())
print(f"[cam] |w2c - inv(c2w)|max = {err_inv:.2e}   (0 => consistent OpenCV w2c/c2w)")

# (b) viser<->OpenCV round-trip through the EXACT demo helper (no basis flip).
try:
    from viewer import viser_camera_to_opencv_viewmat, _rotmat_to_wxyz
    pos  = c2w_B[:3, 3].cpu().numpy()
    wxyz = _rotmat_to_wxyz(c2w_B[:3, :3].cpu().numpy())
    w2c_round = viser_camera_to_opencv_viewmat(pos, wxyz)
    print(f"[cam] viser->OpenCV round-trip |dw2c|max = {np.abs(w2c_round - w2c_B.cpu().numpy()).max():.2e}")
except Exception as e:
    print("[cam] viser round-trip skipped:", type(e).__name__, e)

# (c) SEEN-region depth must back-project onto the existing splats (proof camera+depth in W).
pts_seen = unproject(depth_B, K_B, c2w_B)[torch.from_numpy(seen).to(device)].cpu().numpy()
d_seen = nn_dist(pts_seen, base_np)
print(f"[depth] seen-region backproject -> nearest base point: "
      f"median={np.median(d_seen):.4f} mean={d_seen.mean():.4f}  (scene_scale={scene_scale:.3f})")
print("        => ~0 confirms B's camera + seen depth share world frame W.")
show_overlay([(base_np, "0.7", "base scene"), (pts_seen, "tab:green", "B seen-depth backproj")],
             "Section 3: B's seen-region depth lands on the existing scene",
             cams=[(c2w_A.cpu().numpy(), "red", "A"), (c2w_B.cpu().numpy(), "lime", "B")])

In [ ]:
# Section 3b — INTRINSICS must match what produced the geometry.
# Backproject B's seen depth with the TRUE K vs the fov=60 hardcode (demo.py:844,
# inpainter.py:247). A K mismatch shears/shifts geometry out of W.
fy60 = 0.5 * H_B / math.tan(0.5 * math.radians(60.0))
K_fov60 = np.array([[fy60, 0, W_B / 2.0], [0, fy60, H_B / 2.0], [0, 0, 1]], np.float32)
seen_t = torch.from_numpy(seen).to(device)
pts_trueK  = unproject(depth_B, K_B,     c2w_B)[seen_t].cpu().numpy()
pts_wrongK = unproject(depth_B, K_fov60, c2w_B)[seen_t].cpu().numpy()
dt = nn_dist(pts_trueK, base_np); dw = nn_dist(pts_wrongK, base_np)
print(f"true  K: fx={K_B[0,0]:.1f} fy={K_B[1,1]:.1f} cx={K_B[0,2]:.1f} cy={K_B[1,2]:.1f}  -> nn median {np.median(dt):.4f}")
print(f"fov60 K: fx={fy60:.1f} fy={fy60:.1f} cx={W_B/2:.1f} cy={H_B/2:.1f}  -> nn median {np.median(dw):.4f}")
print("=> store B with its TRUE K; the fov=60 hardcode only matches if B's real fov is 60 deg.")

## Section 4 — Run the rendered frame through an image model

Fill the disoccluded RGB. Default uses a **cv2 fallback** so the notebook runs without
the 9B download; set `RUN_IMAGE_MODEL=True` to load the real diffusers inpaint pipeline
exactly as `inpainter._get_pipeline`. **The image model only edits RGB inside the mask —
it provides no geometry**, so every depth/camera check below is identical regardless of
which model fills the hole.

In [ ]:
from PIL import Image
RUN_IMAGE_MODEL = False    # True -> load the real diffusers inpaint pipeline (heavy ~9B)
IMAGE_MODEL_ID  = "black-forest-labs/FLUX.2-klein-base-9B"
INPAINT_PROMPT  = ("Fill in the missing regions consistent with the reference view. "
                   "Maintain scene geometry, lighting, and architecture.")
inpaint_mask = (hole.astype(np.uint8) * 255)          # 255 = paint (matches inpainter.py)

def run_image_model(rgb_u8, mask_u8, ref_rgb_u8=None):
    # Mirrors inpainter._get_pipeline + inference. Falls back to cv2 Telea inpaint if the
    # model is unavailable; the geometry checks are identical either way (RGB-only edit).
    if RUN_IMAGE_MODEL:
        try:
            import diffusers
            mid = IMAGE_MODEL_ID.lower()
            if   "klein"   in mid:                cls = diffusers.Flux2KleinInpaintPipeline
            elif "kontext" in mid:                cls = diffusers.FluxKontextInpaintPipeline
            elif "flux" in mid and "fill" in mid: cls = diffusers.FluxFillPipeline
            elif "flux"    in mid:                cls = diffusers.FluxInpaintPipeline
            else:                                 cls = diffusers.AutoPipelineForInpainting
            dtype = torch.bfloat16 if "flux" in mid else torch.float16
            pipe = cls.from_pretrained(IMAGE_MODEL_ID, torch_dtype=dtype).to("cuda")
            img, msk = Image.fromarray(rgb_u8), Image.fromarray(mask_u8)
            kw = dict(prompt=INPAINT_PROMPT, image=img, mask_image=msk,
                      num_inference_steps=28, guidance_scale=3.5, strength=1.0,
                      height=img.height, width=img.width)
            if ref_rgb_u8 is not None and "Klein" in type(pipe).__name__:
                kw["image_reference"] = Image.fromarray(ref_rgb_u8)
            return np.asarray(pipe(**kw).images[0].convert("RGB"))
        except Exception as e:
            print("image model unavailable -> cv2 fallback:", type(e).__name__, e)
    filled = cv2.inpaint(cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2BGR), mask_u8, 3, cv2.INPAINT_TELEA)
    return cv2.cvtColor(filled, cv2.COLOR_BGR2RGB)

# nearest existing camera = reference view (mirrors inpainter's neighbour pick)
ref_idx = int(np.argmin(np.linalg.norm(
    base_data.c2w[:, :3, 3].cpu().numpy() - c2w_B[:3, 3].cpu().numpy(), axis=1)))
ref_rgb = (base_data.rgb[ref_idx].cpu().numpy() * 255).astype(np.uint8)
rgb_inpainted = run_image_model(rgb_B, inpaint_mask, ref_rgb)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].imshow(rgb_B);          ax[0].set_title("splat render (holes)")
ax[1].imshow(inpaint_mask, cmap="gray"); ax[1].set_title("inpaint mask (255=paint)")
ax[2].imshow(rgb_inpainted);  ax[2].set_title("image-model output")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Section 5 — Recover depth for the painted region (the crux)

The splat render gives **depth=0** in the hole (α=0), so those pixels would collapse to
B's origin. We run **DA3** on the inpainted frame (B's pose is known), fit DA3's scale on
the **overlap** with the existing splats (`s_overlap = median(splat_z / da3_z)`), and build
a **composite depth**: seen → splat depth (already W), hole → DA3 depth × `s_overlap`.

We keep B's **true K**; DA3 depth is z-depth, so only its scale needs grounding.
MoGe-absolute is shown for contrast — it is metric, not base-scale, so it does not line
up unless the whole scene is metric.

In [ ]:
img_B = torch.from_numpy(rgb_inpainted)                 # (H,W,3) uint8
_, da3_depth, da3_K, da3_mask = _da3_infer_depth_intrinsics_single(da3_model, img_B, (H_B, W_B))
da3_depth = da3_depth.cpu().numpy(); da3_K = da3_K.cpu().numpy(); da3_mask = da3_mask.cpu().numpy() > 0.5

overlap   = seen & da3_mask & (da3_depth > 0) & (depth_B > 0)
s_overlap = fit_overlap_scale(depth_B, da3_depth, overlap)
comp_depth, seen_m, hole_m = build_composite_depth(depth_B, alpha_B, da3_depth, s_overlap, ALPHA_THRESH)

# Contrast: MoGe-absolute (demo_server DA3<->MoGe LS fit) is metric, NOT base-scale.
try:
    moge_model = load_moge_model(device)
    _, moge_depth, _, moge_mask = moge_infer_depth_intrinsics(
        moge_model, img_B.cpu(), depth_pred_hw=(H_B, W_B), target_hw=(H_B, W_B))
    moge_model.cpu()
    moge_depth = moge_depth.cpu().numpy(); moge_mask = moge_mask.cpu().numpy() > 0.5
    ov_dm = da3_mask & moge_mask & (da3_depth > 0) & (moge_depth > 0) & (moge_depth < 999)
    inv_da3, inv_moge = 1.0/(da3_depth[ov_dm]+1e-6), 1.0/(moge_depth[ov_dm]+1e-6)
    s_da3_to_moge = float((inv_da3*inv_moge).sum() / (inv_da3*inv_da3).sum())
except Exception as e:
    s_da3_to_moge = float("nan"); print("MoGe contrast skipped:", type(e).__name__, e)

print(f"overlap px    = {int(overlap.sum())}")
print(f"s_overlap     = {s_overlap:.4f}   (DA3 -> base via splat-overlap)   <- USE THIS")
print(f"s_da3_to_moge = {s_da3_to_moge:.4f}   (MoGe-absolute/metric; wrong unless scene is metric)")
print(f"da3_K vs true K: dfx={abs(float(da3_K[0,0])-float(K_B[0,0])):.1f} "
      f"dfy={abs(float(da3_K[1,1])-float(K_B[1,1])):.1f}  "
      f"(store TRUE K_B; DA3 depth is z-depth, only its scale is grounded)")

## Section 6 — Exhaustive depth verification in W

**(1)** the **naive** path (splat depth, hole=0) collapses the hole onto B's origin; **(2)**
the **composite** path: seen pixels still land on the scene (nn≈0), hole pixels are
**finite**, **in front of the camera** (z>0), at scene-scale radius (not 0), and **(3)**
**seam-continuous** across the inpaint boundary.

In [ ]:
B_center = c2w_B[:3, 3].cpu().numpy()

# (1) NAIVE path (today's bug): splat depth, hole=0 -> hole collapses to B's origin.
naive_depth = depth_B.copy(); naive_depth[hole_m] = 0.0
pts_naive_hole = unproject(naive_depth, K_B, c2w_B)[torch.from_numpy(hole_m).to(device)].cpu().numpy()
collapse = float(np.linalg.norm(pts_naive_hole - B_center, axis=1).mean())

# (2) COMPOSITE path: seen lands on scene, hole is finite / in-front / continuous.
world_comp    = unproject(comp_depth, K_B, c2w_B)
pts_comp_seen = world_comp[torch.from_numpy(seen_m).to(device)].cpu().numpy()
pts_comp_hole = world_comp[torch.from_numpy(hole_m).to(device)].cpu().numpy()
d_comp_seen   = nn_dist(pts_comp_seen, base_np)
d_comp_hole   = nn_dist(pts_comp_hole, base_np)
hole_z        = comp_depth[hole_m]
finite_ok     = bool(np.isfinite(pts_comp_hole).all())
front_ok      = bool((hole_z > 0).all())
hole_radius   = float(np.linalg.norm(pts_comp_hole - B_center, axis=1).mean())

# (3) seam continuity across the inpaint boundary (median depth just inside vs outside).
ring_in  = hole_m & ~ndi.binary_erosion(hole_m, iterations=3)
ring_out = ndi.binary_dilation(hole_m, iterations=3) & seen_m
seam_in  = float(np.median(comp_depth[ring_in]))  if ring_in.any()  else float("nan")
seam_out = float(np.median(comp_depth[ring_out])) if ring_out.any() else float("nan")
seam_ratio = seam_in / max(seam_out, 1e-6)

print(f"(1) naive hole -> B origin: mean |p - B| = {collapse:.4f}  (~0 => COLLAPSE, the bug)")
print(f"(2) composite seen nn-dist: median={np.median(d_comp_seen):.4f}  (~0 => in W)")
print(f"    composite hole nn-dist: median={np.median(d_comp_hole):.4f}  (finite, scene-scale, NOT origin)")
print(f"    hole finite={finite_ok}  in-front(z>0)={front_ok}  hole mean|p-B|={hole_radius:.4f}")
print(f"(3) seam continuity: depth_in={seam_in:.3f} depth_out={seam_out:.3f} ratio={seam_ratio:.3f} (~1 => continuous)")

ns = min(4000, len(pts_comp_seen))
sub_seen = pts_comp_seen[np.random.choice(len(pts_comp_seen), ns, replace=False)] if ns > 0 else pts_comp_seen
show_overlay([(base_np, "0.8", "base"),
              (pts_naive_hole, "tab:red",   "naive hole -> B origin"),
              (pts_comp_hole,  "tab:green", "composite hole (DA3*overlap)"),
              (sub_seen,       "tab:blue",  "composite seen")],
             "Section 6: composite fills the hole in W; naive collapses to origin",
             cams=[(c2w_B.cpu().numpy(), "lime", "B")])

## Section 7 — Append B into the frameset + cameras, then re-verify

Append `(rgb=inpainted, K=K_B, c2w=c2w_B, depth=composite)` via the real
`VideoData.append_frame` (what `SplatTrainer.append_supplied_frames` calls), and build the
`train_mask` with `_per_frame_train_mask`. Then re-verify **from the stored frameset**
using the trainer's *own* unprojection: `w2c==inv(c2w)`, the stored seen-depth still lands
in W, `train_mask` supervises the painted region, and the epoch tag is bumped.

In [ ]:
rgb_app   = torch.from_numpy(rgb_inpainted.astype(np.float32) / 255.0).to(device)
depth_app = torch.from_numpy(comp_depth.astype(np.float32)).to(device)
conf_app  = torch.ones((H_B, W_B), device=device)               # painted depth: supervise all
sky_app   = torch.zeros((H_B, W_B), dtype=torch.bool, device=device)

n_before = base_data.N
new_idx  = base_data.append_frame(rgb_app, depth_app, K_B.clone(), c2w_B.clone(),
                                  conf=conf_app, sky=sky_app,
                                  epoch=(max(base_data.frame_epoch) + 1))
train_mask_new = _per_frame_train_mask(depth_app, sky_app, conf_app, None, True)   # as append_supplied_frames
print(f"appended frame -> idx {new_idx} (N {n_before}->{base_data.N}), epoch {base_data.frame_epoch[-1]}")

# Production equivalent on a LIVE SplatTrainer `t` (same payload, same result):
#   t.append_supplied_frames([{ "rgb": rgb_inpainted,
#                               "K":   K_B.cpu().numpy(),
#                               "c2w": c2w_B.cpu().numpy(),
#                               "depth": comp_depth }], seed_new_splats=True)

# --- Re-verify straight from the STORED frameset using the trainer's OWN math ---
i = new_idx
err_store = float((base_data.w2c[i] - torch.linalg.inv(base_data.c2w[i])).abs().max())
world_store = unproject(base_data.depth[i].cpu().numpy(),
                        base_data.K[i].cpu().numpy(),
                        base_data.c2w[i].cpu().numpy())          # trainer's exact unproject
d_store_seen = nn_dist(world_store[torch.from_numpy(seen_m).to(device)].cpu().numpy(), base_np)
tm = train_mask_new.cpu().numpy()
mask_covers_hole = bool(tm[hole_m].all())
depth_pos_hole   = bool((base_data.depth[i].cpu().numpy()[hole_m] > 0).all())
print(f"[store] |w2c - inv(c2w)|max = {err_store:.2e}")
print(f"[store] stored-frame seen backproject nn-dist median = {np.median(d_store_seen):.4f}  (matches Section 6)")
print(f"[store] train_mask supervises hole = {mask_covers_hole}   stored hole depth>0 = {depth_pos_hole}")
print(f"[store] frame_epoch tail = {base_data.frame_epoch[-3:]}  (new session tag bumped)")

## Section 8 — Seed-style unprojection (what Phase-2 seeding does)

`_seed_splats_for_new_frames` unprojects the appended frame's valid pixels into W with the
identical einsum. Confirm the **painted region becomes splats that join the scene**, not a
spray at the camera origin.

In [ ]:
world_seed = unproject(base_data.depth[i].cpu().numpy(),
                       base_data.K[i].cpu().numpy(),
                       base_data.c2w[i].cpu().numpy())
valid_seed = (base_data.depth[i] > 0).cpu().numpy() & tm
pts_seed_hole = world_seed[torch.from_numpy(hole_m & valid_seed).to(device)].cpu().numpy()
d_seed_hole = nn_dist(pts_seed_hole, base_np)
print(f"seeded points: {int(valid_seed.sum())}  (painted-region {int((hole_m & valid_seed).sum())})")
print(f"painted-region seeded points -> nearest base point median = {np.median(d_seed_hole):.4f}")
show_overlay([(base_np, "0.8", "base scene"),
              (pts_seed_hole, "tab:green", "new splats (painted region)")],
             "Section 8: appended frame seeds splats that join the scene (no origin spray)",
             cams=[(c2w_B.cpu().numpy(), "lime", "B")])

## Conclusions — production checklist

For a new view to enter training in the right coordinate frame **W**:

1. **Camera** — author B's `c2w` directly in W; store **OpenCV `w2c = inv(c2w)`** and B's
   **true `K`** (never the `fov=60°` hardcode unless that is B's real fov). Section 3 checks
   all three.
2. **Seen-region depth** — the splat-rendered depth (`ED/alpha`) is already in W; it
   back-projects onto the existing splats (Section 3c).
3. **Painted-region depth** — do **NOT** keep the splat depth (it is 0 → origin collapse).
   Re-estimate with **DA3** and scale on the **splat overlap** (`s_overlap`); build the
   **composite depth** (Section 5). This is the single fix that makes the new frame's
   geometry match W.
4. **Append** — push `(rgb, K_B, c2w_B, composite_depth)` into the frameset and supervise
   the painted region in `train_mask` (Section 7).

Production call on a live trainer `t`:

```python
t.append_supplied_frames([{ "rgb":   rgb_inpainted,
                            "K":     K_B.cpu().numpy(),
                            "c2w":   c2w_B.cpu().numpy(),
                            "depth": comp_depth }],
                         seed_new_splats=True)
```

**Caveat surfaced (Section 5):** DA3 predicts its own `K`; we store B's true `K` and only
borrow DA3's z-depth (scaled). If `da3_K` differs a lot from `K_B`, warp DA3 depth into
`K_B`'s pixel grid before compositing — the printed `dfx/dfy` tells you whether that matters
for this scene. Today's `inpainter._on_add_frame_click` appends the **raw splat depth**
(Section 6 path 1), which is the documented origin-collapse bug; this notebook is the
corrected forward path.

## Verified result — yes, we have a camera that works

Ran end-to-end in the `lyra2` env (museum_fwd.mp4, N=10 base frames, scene_scale=2.192,
new camera B with a 49% disocclusion). **The new camera lands in the same world frame W as
the existing frames** — provided it is appended with its **true K** and a **composite depth**.
Measured:

| Check | Result | What it proves |
|---|---|---|
| `\|w2c − inv(c2w)\|` max | **0.00e+00** | camera matrices self-consistent (OpenCV) |
| viser→OpenCV round-trip | **1.05e-07** | the demo's camera path adds no basis flip |
| seen-region depth → base splats (nn) | **median 0.037** (≈1.7% of scene_scale) | B's pose + seen depth are in W |
| true K vs `fov=60°` K (nn) | **0.037 vs 1.12** | wrong K shears geometry ~30× out of W |
| naive hole depth → B origin | **0.0000** | raw splat depth collapses the disocclusion to the camera centre (the bug) |
| composite hole: finite / z>0 / seam ratio | **True / True / 0.94** | DA3-overlap depth fills the hole continuously, in front of the camera |
| overlap scale used | **s_overlap = 1.58** (vs MoGe-absolute 2.51) | DA3 depth grounded to the scene, not to metric units |
| after append, stored frame re-check | **`w2c==inv(c2w)`=0, seen nn 0.037, train_mask covers hole, epoch→1** | the frame entered the frameset correctly in W |
| painted-region seeded splats → scene (nn) | **median 0.69** | new splats *join* the scene, no origin spray |

**Bottom line.** Append `(rgb, **true K**, c2w, **composite_depth**)` and the camera is correct
in W. The two non-negotiables: **(1)** store the camera's **true K**, never the `fov=60°`
hardcode unless that is its real fov; **(2)** depth in the painted region must be **DA3
re-estimated and scaled on the splat overlap** (the composite), not the raw splat-rendered
depth.

> ⚠️ **Demo gap.** The live `inpainter._on_add_frame_click` still appends the **raw splat
> depth** (the `0.0000`-collapse row above), so a camera added through the running demo today
> is *not* yet correct. This notebook's composite-depth path is the fix that needs porting into
> that handler.